In [2]:
import os
import cv2
import dlib
import numpy as np
from moviepy.editor import VideoFileClip, AudioFileClip, concatenate_videoclips

# Paths to input and output directories
input_dir = "datasetnew/"
output_base_dir = "cropped_lips_new/"
output_annotated_dir = "annotated_videos_new/"

# Load Dlib's pre-trained face detector and the facial landmarks predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")  # Ensure this file is in the correct path

RuntimeError: Unable to open shape_predictor_68_face_landmarks.dat

In [2]:
# Function to detect lips and crop around them
def crop_lips(video_path, output_dir, output_annotated_path, target_size=(256, 128)):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file {video_path}")
        return

    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Original video ({video_path}): {total_frames} frames")

    lips_centers = []  # List to store the centers of the lips regions

    frame_index = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Verify frame is valid
        if frame is None or frame.size == 0:
            print(f"Invalid frame captured from {video_path}")
            continue

        # Convert frame to grayscale for facial landmarks detection
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Detect faces in the frame
        faces = detector(gray)

        if len(faces) > 0:
            # Assume the first detected face is the target face
            face = faces[0]

            # Get the landmarks for the face
            landmarks = predictor(gray, face)

            # Extract coordinates for the lips region
            x_coords = [landmarks.part(i).x for i in range(48, 68)]
            y_coords = [landmarks.part(i).y for i in range(48, 68)]

            # Calculate the center of the lips region
            center_x = int(np.mean(x_coords))
            center_y = int(np.mean(y_coords))

            # Store the center of the lips region and frame
            lips_centers.append((center_x, center_y, frame))

        else:
            print(f"No faces detected in frame from {video_path}")

        frame_index += 1

    # Crop frames to the target size around the center of the lips
    cropped_frames = []
    for center_x, center_y, frame in lips_centers:
        x_min = max(0, center_x - target_size[0] // 2)
        x_max = min(frame.shape[1], center_x + target_size[0] // 2)
        y_min = max(0, center_y - target_size[1] // 2)
        y_max = min(frame.shape[0], center_y + target_size[1] // 2)

        # Ensure the cropped frame fits within the bounds of the original frame
        if x_max - x_min != target_size[0]:
            if x_max == frame.shape[1]:
                x_min = x_max - target_size[0]
            else:
                x_max = x_min + target_size[0]

        if y_max - y_min != target_size[1]:
            if y_max == frame.shape[0]:
                y_min = y_max - target_size[1]
            else:
                y_max = y_min + target_size[1]

        # Crop the frame
        cropped_frame = frame[y_min:y_max, x_min:x_max]

        cropped_frames.append(cropped_frame)

    # Initialize video writer for cropped lips
    output_video_path = os.path.join(output_dir, os.path.basename(video_path))
    temp_video_path = os.path.join(output_dir, "temp_" + os.path.basename(video_path))
    out = cv2.VideoWriter(temp_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, target_size)

    # Write all cropped frames to output video
    for frame in cropped_frames:
        out.write(frame)

    cropped_total_frames = len(cropped_frames)
    print(f"Cropped video ({output_video_path}): {cropped_total_frames} frames")

    # Release everything
    cap.release()
    out.release()

    # Add audio to the cropped video
    original_video = VideoFileClip(video_path)
    cropped_video = VideoFileClip(temp_video_path)
    cropped_video = cropped_video.set_audio(original_video.audio)
    cropped_video.write_videofile(output_video_path, codec='libx264', audio_codec='aac')

    # Remove the temporary video file
    os.remove(temp_video_path)

    print(f"Cropped lips with audio saved to {output_video_path}")

In [3]:
# Iterate through each user's directory
for user_dir in os.listdir(input_dir):
    user_path = os.path.join(input_dir, user_dir)
    if os.path.isdir(user_path):
        # Create output directory for cropped lips videos
        output_user_dir = os.path.join(output_base_dir, user_dir)
        os.makedirs(output_user_dir, exist_ok=True)

        # Create output directory for annotated videos
        output_annotated_user_dir = os.path.join(output_annotated_dir, user_dir)
        os.makedirs(output_annotated_user_dir, exist_ok=True)

        # Iterate through each video file in the user's directory
        for video_file in os.listdir(user_path):
            if video_file.lower().endswith(".mp4"):
                video_path = os.path.join(user_path, video_file)

                # Process and crop lips from the video
                crop_lips(video_path, output_user_dir, output_annotated_user_dir)

Original video (datasetnew/Carlo\VID_20240702_155539_computational.mp4): 58 frames
Cropped video (cropped_lips_new/Carlo\VID_20240702_155539_computational.mp4): 58 frames
Moviepy - Building video cropped_lips_new/Carlo\VID_20240702_155539_computational.mp4.
MoviePy - Writing audio in VID_20240702_155539_computationalTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video cropped_lips_new/Carlo\VID_20240702_155539_computational.mp4



Moviepy - Done !
Moviepy - video ready cropped_lips_new/Carlo\VID_20240702_155539_computational.mp4
Cropped lips with audio saved to cropped_lips_new/Carlo\VID_20240702_155539_computational.mp4
Original video (datasetnew/Carlo\VID_20240702_155540_computational.mp4): 66 frames
Cropped video (cropped_lips_new/Carlo\VID_20240702_155540_computational.mp4): 61 frames
Moviepy - Building video cropped_lips_new/Carlo\VID_20240702_155540_computational.mp4.
MoviePy - Writing audio in VID_20240702_155540_computationalTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video cropped_lips_new/Carlo\VID_20240702_155540_computational.mp4



Moviepy - Done !
Moviepy - video ready cropped_lips_new/Carlo\VID_20240702_155540_computational.mp4
Cropped lips with audio saved to cropped_lips_new/Carlo\VID_20240702_155540_computational.mp4


...

Original video (datasetnew/Carlo\VID_20240702_155612_smartphone.mp4): 78 frames
Cropped video (cropped_lips_new/Carlo\VID_20240702_155612_smartphone.mp4): 68 frames
Moviepy - Building video cropped_lips_new/Carlo\VID_20240702_155612_smartphone.mp4.
MoviePy - Writing audio in VID_20240702_155612_smartphoneTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video cropped_lips_new/Carlo\VID_20240702_155612_smartphone.mp4



Moviepy - Done !
Moviepy - video ready cropped_lips_new/Carlo\VID_20240702_155612_smartphone.mp4
Cropped lips with audio saved to cropped_lips_new/Carlo\VID_20240702_155612_smartphone.mp4
Original video (datasetnew/Carlo\VID_20240702_155613_smartphone.mp4): 81 frames
Cropped video (cropped_lips_new/Carlo\VID_20240702_155613_smartphone.mp4): 69 frames
Moviepy - Building video cropped_lips_new/Carlo\VID_20240702_155613_smartphone.mp4.
MoviePy - Writing audio in VID_20240702_155613_smartphoneTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video cropped_lips_new/Carlo\VID_20240702_155613_smartphone.mp4



Moviepy - Done !
Moviepy - video ready cropped_lips_new/Carlo\VID_20240702_155613_smartphone.mp4
Cropped lips with audio saved to cropped_lips_new/Carlo\VID_20240702_155613_smartphone.mp4
